# Shor算法的经典实现

我们首先写出如下等比数列：

In [20]:
for i in range(11):
    print(2 ** i, end=' ')

1 2 4 8 16 32 64 128 256 512 1024 

接下来我们让其对15取余

In [2]:
for i in range(11):
    print(2 ** i % 15, end=', ')

1, 2, 4, 8, 1, 2, 4, 8, 1, 2, 4, 

啊哦，我们好像观察到了某种周期性序列！观察可知该序列以4为周期。我们不妨换个数字试一试：

In [3]:
for i in range(11):
    print(2 ** i % 21, end=', ')

1, 2, 4, 8, 16, 11, 1, 2, 4, 8, 16, 

这时它的周期为6。好了，我们开始怀疑这是否是某种通用的数学定理了。事实上，欧拉在18世纪60年代发现了如上美丽的模式。

设N是两个素数$p$和$q$的乘积，并考虑这个数列

$$x \mod N,\quad x^2 \mod N,\quad x^3 \mod N,\quad x^4 \mod N, ...$$

那么，如果$x$不能被$p$或$q$整除，则上述序列将以一段能整除$(p-1)(q-1)$的周期数列。

对于$N=15=3\times 5$，$(p-1)(q-1)=8$可以整除4！

对于$N=21=3\times 7$，$(p-1)(q-1)=12$可以整除6！

延续这一思路，我们可以尝试利用欧拉的这个定理实现大整数的因子分解，亦即Shor算法。

经典Shor算法流程如下：
![](10_7.png)

改用数学表达即：
![](10_8.png)

代码实现如下

In [4]:
import random
import math
import itertools
def period_finding_classical(a,N):
    for r in itertools.count(start=1):
        if (a**r) % N == 1:
            return r

def shors_algorithm_classical(N):
    assert(N>0)
    assert(int(N)==N)
    while True:
        a=random.randint(0,N-1)
        g=math.gcd(a,N)
        if g!=1 or N==1:
            first_factor=g
            second_factor=int(N/g)
            return first_factor,second_factor
        else:
            r=period_finding_classical(a,N)  
            if r % 2 != 0:
                continue
            elif a**(int(r/2)) % N == -1 % N:
                continue
            else:
                first_factor=math.gcd(a**int(r/2)+1,N)
                second_factor=math.gcd(a**int(r/2)-1,N)
                if first_factor==N or second_factor==N:
                    continue
                return first_factor,second_factor

In [5]:
shors_algorithm_classical(15)

(3, 5)

In [6]:
shors_algorithm_classical(91)

(7, 13)

In [23]:
shors_algorithm_classical(38)

(2, 19)

# 量子Shor算法（以$N=15,a=2$为例）
我们采用改进的电路进行演示
![](10_10.png)
## 利用数学技巧取模
在量子计算机上取模并不容易，一种简单粗暴的思路是模仿取余的汇编代码，之后用量子逻辑门逐一实现，然而这种方式需要大量量子比特，自然会导致我们的算法不够稳定。接下来我们介绍一种数学技巧，它把**求余运算**化简成了**对应问题**。

注意到，由于我们对$N=15$取余，所以结果只能在$0\sim14$当中。我们以1开始观察以下数列：
$$2 \times 1 \mod 15 = 2$$
$$2 \times 2 \mod 15 = 4$$
$$2 \times 4 \mod 15 = 8$$
$$2 \times 8 \mod 15 = 1$$

啊哦，我们好像又找到了一个周期数列（喜）。我们换成3开始再试试：

$$2 \times 3 \mod 15 = 6$$
$$2 \times 6 \mod 15 = 12$$
$$2 \times 12 \mod 15 = 9$$
$$2 \times 9 \mod 15 = 3$$

事实上，对于所有的$2x(\mod 15)$来说，所有可能的循环如下所示：

$$1\rightarrow2\rightarrow4\rightarrow8\rightarrow1$$
$$3\rightarrow6\rightarrow12\rightarrow9\rightarrow3$$
$$5\rightarrow10\rightarrow5$$
$$7\rightarrow14\rightarrow13\rightarrow11\rightarrow7$$

也就是说，所谓的求余运算可以等价成“查字典”一样的操作。更神奇的是，当我们把十进制改为二进制时，会有新的模式出现！

$$0001\rightarrow0010\rightarrow0100\rightarrow1000\rightarrow0001$$
$$0011\rightarrow0110\rightarrow1100\rightarrow1001\rightarrow0011$$
$$0101\rightarrow1010\rightarrow1010$$
$$0111\rightarrow1110\rightarrow1101\rightarrow1011\rightarrow0111$$

这比“查字典”操作更为简单明了，我们只需要每次向左挪动一个二进制位就实现了求余！

更细致地，我们可以把左移一位表述为：
+ 交换第3个量子比特和第0个量子比特
+ 交换第0个量子比特和第1个量子比特
+ 交换第1个量子比特和第2个量子比特

In [7]:
import qiskit
import matplotlib
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit import BasicAer

def mult_2mod15_quantum(qr,qc):
    # Swap 0th qubit and 3rd qubit
    qc.swap(qr[0], qr[3])

    # Swap 0th qubit and 1st qubit
    qc.swap(qr[1],qr[0])

    # Swap 1st qubit and 2nd qubit
    qc.swap(qr[1],qr[2])

为了后续算法的执行，我们需要将其改装成受控门

In [8]:
def controlled_mult_2mod15_quantum(qr,qc,control_qubit):
    """
    Controlled quantum circuit for multiplication by 2 mod 15.
        Note: control qubit should an index greater than 3, 
        and qubits 0,1,2,3 are reserved for circuit operations
    """
    # Swap 0th qubit and 3rd qubit
    qc.cswap(control_qubit,qr[0],qr[3])

    # Swap 0th qubit and 1st qubit
    qc.cswap(control_qubit,qr[1],qr[0])

    # Swap 1st qubit and 2nd qubit
    qc.cswap(control_qubit,qr[1],qr[2])


## 剩余电路部分
![](10_9.png)

In [10]:
import math
def shors_subroutine_period_2mod15(qr,qc,cr):
    qc.x(qr[0])
    qc.h(qr[4])
    qc.h(qr[4])
    qc.measure(qr[4],cr[0])

    qc.h(qr[5])
    qc.cx(qr[5],qr[0])
    qc.cx(qr[5],qr[2])
    if cr[0] == 1:
        qc.u1(math.pi/2,qr[4]) #pi/2 is 90 degrees in radians
    qc.h(qr[5])
    qc.measure(qr[5],cr[1])

    qc.h(qr[6])
    controlled_mult_2mod15_quantum(qr,qc,qr[6])
    if cr[1] == 1:
        qc.u1(math.pi/2,qr[6]) # pi/2 is 90 degrees in radians
    if cr[0] == 1:
        qc.u1(math.pi/4,qr[6]) #pi/4 is 45 degrees in radians
    qc.h(qr[6])
    qc.measure(qr[6],cr[2]) 

## 识别周期
我们除了利用量子计算的独特性外，还需要正确地识别出周期。我们这里再介绍一种数学技巧，它使我们甚至不需要求余，就可以知道它的周期！

这种方法就是**连分数展开**。对于任何正有理数$\xi$，我们知道可以展开成如下形式：
![](10_11.png)
其中系数满足：
![](10_12.png)
我们还可以重写每个$\xi_n$，使其写成$\xi_n=p_n/q_n$，同样也可以写成数列的形式：
$$p_0=a_0,p_1=a_1a_0+1;\qquad p_n=a_np_{n-1}+p_{n-2}$$
$$q_0=1,q_1=a_1;\qquad q_n=a_nq_{n-1}+q_{n-2}$$
如果想获取周期，我们使用连分数展开计算测量结果除以总测量可能性即可，也就是
$$\frac{\text{测量结果}}{2^\text{量子比特数}}$$


我们假设电路由三个量子比特组成，最终测量结果假设得到6，由于总可能性为$2^3=8$种，我们得到分数$\xi = 6/8$，简单的计算告诉我们：
$$\frac{p_0}{q_0}=\frac{0}{1},\frac{p_1}{q_1}=\frac{1}{1},\frac{p_2}{q_2}=\frac{3}{4}$$
注意到最后一个分数的分母为4，数学中的一个小定理告诉我们这意味着周期为4（符合我们先前的观测）！

### 于是我们可以利用连分数展开，实现从量子寄存器中直接读出周期！

In [11]:
# see https://arxiv.org/pdf/quant-ph/0010034.pdf for more details (convergence relations on page 11)
import math
import numpy
def continued_fraction(xi,max_steps=100): # stop_after is cutoff for algorithm, for debugging
    """
    This function computes the continued fraction expansion of input xi
    per the recurrance relations on page 11 of https://arxiv.org/pdf/quant-ph/0010034.pdf
    
    """
    #a and xi initial
    all_as=[]
    all_xis=[]
    a_0=math.floor(xi)
    xi_0=xi-a_0
    all_as+=[a_0]
    all_xis+=[xi_0]
    # p and q initial
    all_ps=[]
    all_qs=[]
    p_0=all_as[0]
    q_0=1
    all_ps+=[p_0]
    all_qs+=[q_0]
    
    xi_n=xi_0
    while not numpy.isclose(xi_n,0,atol=1e-7):
        if len(all_as)>=max_steps:
            print("Warning: algorithm did not converge within max_steps %d steps, try increasing max_steps" % max_steps)
            break
        # computing a and xi
        a_nplus1=math.floor(1/xi_n)
        xi_nplus1=1/xi_n-a_nplus1
        all_as+=[a_nplus1]
        all_xis+=[xi_nplus1]
        xi_n=xi_nplus1
        # computing p and q
        n=len(all_as)-1
        if n==1:
            p_1=all_as[1]*all_as[0]+1
            q_1=all_as[1]
            all_ps+=[p_1]
            all_qs+=[q_1]
        else:
            p_n=all_as[n]*all_ps[n-1]+all_ps[n-2]
            q_n=all_as[n]*all_qs[n-1]+all_qs[n-2]
            all_ps+=[p_n]
            all_qs+=[q_n]
    return all_ps,all_qs,all_as,all_xis

from IPython.display import display, Math
def pretty_print_continued_fraction(results,raw_latex=False):
    all_ps,all_qs,all_as,all_xis=results
    for i,vals in enumerate(zip(all_ps,all_qs,all_as,all_xis)):
        p,q,a,xi=vals
        if raw_latex:
            print(r'\frac{p_%d}{q_%d}=\frac{%d}{%d}'%(i,i,p,q))
        else:
            display(Math(r'$\frac{p_%d}{q_%d}=\frac{%d}{%d}$'%(i,i,p,q)))
    
pretty_print_continued_fraction(continued_fraction(6/8))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

# 下一步我们把连分数展开算法整合到Shor子例程当中

In [12]:
import math
def period_from_quantum_measurement(quantum_measurement,
                                    number_qubits,
                                    a_shor,
                                    N_shor,
                                    max_steps=100): # stop_after is cutoff for algorithm, for debugging
    """
    This function computes the continued fraction expansion of input xi
    per the recurrance relations on page 11 of https://arxiv.org/pdf/quant-ph/0010034.pdf
    a_shor is the random number chosen as part of Shor's algorithm
    N_shor is the number Shor's algorithm is trying to factor
    """
    xi=quantum_measurement/2**number_qubits
    
    #a and xi initial
    all_as=[]
    all_xis=[]
    a_0=math.floor(xi)
    xi_0=xi-a_0
    all_as+=[a_0]
    all_xis+=[xi_0]
    # p and q initial
    all_ps=[]
    all_qs=[]
    p_0=all_as[0]
    q_0=1
    all_ps+=[p_0]
    all_qs+=[q_0]
    
    xi_n=xi_0
    while not numpy.isclose(xi_n,0,atol=1e-7):
        if len(all_as)>=max_steps:
            print("Warning: algorithm did not converge within max_steps %d steps, try increasing max_steps" % max_steps)
            break
        # computing a and xi
        a_nplus1=math.floor(1/xi_n)
        xi_nplus1=1/xi_n-a_nplus1
        all_as+=[a_nplus1]
        all_xis+=[xi_nplus1]
        xi_n=xi_nplus1
        # computing p and q
        n=len(all_as)-1
        if n==1:
            p_1=all_as[1]*all_as[0]+1
            q_1=all_as[1]
            all_ps+=[p_1]
            all_qs+=[q_1]
        else:
            p_n=all_as[n]*all_ps[n-1]+all_ps[n-2]
            q_n=all_as[n]*all_qs[n-1]+all_qs[n-2]
            all_ps+=[p_n]
            all_qs+=[q_n]
        # check the q to see if it is our answer (note with this we skip the first q, as a trivial case)
        if a_shor**all_qs[-1]%N_shor == 1 % N_shor:
            return all_qs[-1]

period_from_quantum_measurement(13453,14,3,91) #should return, for example 6 per page 20 of https://arxiv.org/pdf/quant-ph/0010034.pdf

6

In [13]:
# Testing this:
import qiskit
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister

def binary_string_to_decimal(s):
    dec=0
    for i in s[::-1]:
        if int(i):
            dec+=2**int(i)
    return dec

def run_shors_subroutine_period2_mod15():
    qr = QuantumRegister(7)
    cr = ClassicalRegister(3)
    qc = QuantumCircuit(qr,cr)
    # initialize x to be a superposition of all possible r quibit values
    #for i in range(4):
    #    qc.h(qr[i])
    # run circuit (which includes measurement steps)
    shors_subroutine_period_2mod15(qr,qc,cr)
        
    import time
    from qiskit.tools.visualization import plot_histogram
    backend=BasicAer.get_backend('qasm_simulator')
    job_exp = qiskit.execute(qc, backend=backend,shots=1)
    result = job_exp.result()
    final=result.get_counts(qc)
    # convert final result to decimal
    measurement=binary_string_to_decimal(list(final.keys())[0])
    period_r=period_from_quantum_measurement(measurement,3,2,15)
    return period_r
print(run_shors_subroutine_period2_mod15())

4


## 最后一步：把所有模块整合成完整的Shor算法
注意周期函数是用来找**指数**的周期的！

In [14]:
def period_finding_quantum(a,N):
    # for the sake of example we will not implement this algorithm in full generality
    # rather, we will create an example with one specific a and one specific N
    # extension work could be done to impl
    if a==2 and N==15:
        return run_shors_subroutine_period2_mod15()
    else:
        raise Exception("Not implemented for N=%d, a=%d" % (N,a))
        
def shors_algorithm_quantum(N,fixed_a=None):
    assert(N>0)
    assert(int(N)==N)
    while True:
        if not fixed_a:
            a=random.randint(0,N-1) 
        else:
            a=fixed_a
        g=math.gcd(a,N)
        if g!=1 or N==1:
            first_factor=g
            second_factor=int(N/g)
            return first_factor,second_factor
        else:
            r=period_finding_quantum(a,N)  
            if not r:
                continue
            if r % 2 != 0:
                continue
            elif a**(int(r/2)) % N == -1 % N:
                continue
            else:
                first_factor=math.gcd(a**int(r/2)+1,N)
                second_factor=math.gcd(a**int(r/2)-1,N)
                if first_factor==N or second_factor==N:
                    continue
                if first_factor*second_factor!=N:
                    # checking our work
                    continue
                return first_factor,second_factor

In [24]:
# Here's our final result
shors_algorithm_quantum(15,fixed_a=2)

(5, 3)

In [16]:
# Now trying it out to see how the algorithm would function if we let it choose a given random a:
for a in range(15):
    # Here's the result for a given a:
    try:
        print("randomly chosen a=%d would result in %s"%(a,shors_algorithm_quantum(15,fixed_a=a)))
    except:
        print("FINISH IMPLEMENTING algorithm doesn't work with a randomly chosen a=%d at this stage"%a)

FINISH IMPLEMENTING algorithm doesn't work with a randomly chosen a=0 at this stage
FINISH IMPLEMENTING algorithm doesn't work with a randomly chosen a=1 at this stage
randomly chosen a=2 would result in (5, 3)
randomly chosen a=3 would result in (3, 5)
FINISH IMPLEMENTING algorithm doesn't work with a randomly chosen a=4 at this stage
randomly chosen a=5 would result in (5, 3)
randomly chosen a=6 would result in (3, 5)
FINISH IMPLEMENTING algorithm doesn't work with a randomly chosen a=7 at this stage
FINISH IMPLEMENTING algorithm doesn't work with a randomly chosen a=8 at this stage
randomly chosen a=9 would result in (3, 5)
randomly chosen a=10 would result in (5, 3)
FINISH IMPLEMENTING algorithm doesn't work with a randomly chosen a=11 at this stage
randomly chosen a=12 would result in (3, 5)
FINISH IMPLEMENTING algorithm doesn't work with a randomly chosen a=13 at this stage
FINISH IMPLEMENTING algorithm doesn't work with a randomly chosen a=14 at this stage
